In [1]:
# =========================
# 0) Install (Colab)
# =========================
!pip install -U torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -U unsloth
!pip install -U transformers==4.56.2 datasets==4.3.0
!pip install -U --no-deps trl==0.22.2

import torch
assert torch.cuda.is_available(), "Enable GPU runtime (Colab → Runtime → GPU)"

Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 68.4 MB/s eta 0:00:00

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 13.7 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0


In [2]:
# =========================
# 1) OOP: Unsloth FineTuner
# =========================
from dataclasses import dataclass
from typing import Optional, List
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import SFTTrainer, SFTConfig


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
@dataclass
class FineTuneConfig:
    # model
    model_name: str
    load_in_4bit: bool = True
    max_seq_length: int = 4096
    dtype: Optional[str] = None  # None = auto

    # dataset
    dataset_name: str = "tatsu-lab/alpaca"
    split: str = "train"
    seed: int = 3407

    # training
    output_dir: str = "outputs"
    lora_save_path: str = "lora_adapters"
    per_device_bs: int = 2
    grad_acc_steps: int = 4
    epochs: int = 1
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    logging_steps: int = 10
    packing: bool = True

    # lora
    lora_r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    target_modules: List[str] = None
    use_gc: bool = False

    # save merged model
    save_merged: bool = False
    merged_save_path: str = "merged_fp16_model"

In [4]:
class UnslothFineTuner:
    def __init__(self, cfg: FineTuneConfig):
        self.cfg = cfg
        if self.cfg.target_modules is None:
            self.cfg.target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
        self.model = None
        self.tokenizer = None

    # ---------- Load Model ----------
    def load_model(self):
        print(f"✅ Loading model: {self.cfg.model_name}")

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.cfg.model_name,
            max_seq_length=self.cfg.max_seq_length,
            dtype=self.cfg.dtype,
            load_in_4bit=self.cfg.load_in_4bit,
        )

        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.cfg.lora_r,
            target_modules=self.cfg.target_modules,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            use_gradient_checkpointing=self.cfg.use_gc,
            random_state=self.cfg.seed,
        )

        self.model.print_trainable_parameters()
        print("Is PEFT model?", isinstance(self.model, PeftModel))
        print("Device:", next(self.model.parameters()).device, " | Dtype:", next(self.model.parameters()).dtype)

    # ---------- Load Dataset ----------
    def load_dataset(self):
        print(f"✅ Loading dataset: {self.cfg.dataset_name} (split={self.cfg.split})")

        # Try HF load first, then local
        try:
            ds = load_dataset(self.cfg.dataset_name, split=self.cfg.split)
        except Exception:
            ds = load_dataset(path=self.cfg.dataset_name, split=self.cfg.split)

        ds = ds.shuffle(seed=self.cfg.seed)
        print(ds)
        print("Columns:", ds.column_names)
        return ds

    # ---------- Formatting Helpers ----------
    def _eos(self):
        return self.tokenizer.eos_token or "</s>"

    def _alpaca_prompt(self):
        return """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

    def _format_alpaca(self, batch):
        EOS = self._eos()
        prompt = self._alpaca_prompt()
        texts = []
        inp_list = batch.get("input", [""] * len(batch["instruction"]))
        for ins, inp, out in zip(batch["instruction"], inp_list, batch["output"]):
            texts.append(prompt.format(
                instruction=ins or "",
                input=inp or "",
                output=out or ""
            ) + EOS)
        return {"text": texts}

    def _format_dolly(self, batch):
        EOS = self._eos()
        prompt = self._alpaca_prompt()
        ctx_list = batch.get("context", [""] * len(batch["instruction"]))
        texts = []
        for ins, ctx, resp in zip(batch["instruction"], ctx_list, batch["response"]):
            texts.append(prompt.format(
                instruction=ins or "",
                input=ctx or "",
                output=resp or ""
            ) + EOS)
        return {"text": texts}

    def _format_sharegpt(self, batch):
        EOS = self._eos()
        texts = []

        conv_key = None
        if "conversations" in batch:
            conv_key = "conversations"
        elif "messages" in batch:
            conv_key = "messages"
        else:
            raise ValueError("ShareGPT format needs 'conversations' or 'messages' column.")

        for conv in batch[conv_key]:
            if conv is None:
                texts.append("" + EOS)
                continue

            turns = []
            for t in conv:
                if isinstance(t, dict) and "from" in t and "value" in t:
                    role, content = t["from"], t["value"]
                elif isinstance(t, dict) and "role" in t and "content" in t:
                    role, content = t["role"], t["content"]
                else:
                    continue
                turns.append((role, content))

            chat = ""
            for role, content in turns:
                if role in ["human", "user"]:
                    chat += f"### User:\n{content}\n\n"
                else:
                    chat += f"### Assistant:\n{content}\n\n"

            texts.append(chat.strip() + EOS)

        return {"text": texts}

    def _format_text(self, batch):
        EOS = self._eos()
        if "text" not in batch:
            raise ValueError("Expected a 'text' column.")
        return {"text": [(t or "") + EOS for t in batch["text"]]}

    def _format_pharma_custom(self, batch):
        """
        For sunny199/pharma-instruction-data OR pharma_instruction_data
        Supports common patterns:
        - instruction,input,output
        - question,answer
        - prompt,completion
        - input,output
        """
        EOS = self._eos()
        cols = set(batch.keys())

        if {"instruction", "output"}.issubset(cols):
            return self._format_alpaca(batch)

        if {"question", "answer"}.issubset(cols):
            texts = []
            for q, a in zip(batch["question"], batch["answer"]):
                texts.append(f"### Question:\n{q or ''}\n\n### Answer:\n{a or ''}" + EOS)
            return {"text": texts}

        if {"prompt", "completion"}.issubset(cols):
            texts = []
            for p, c in zip(batch["prompt"], batch["completion"]):
                texts.append(f"{p or ''}\n{c or ''}" + EOS)
            return {"text": texts}

        if {"input", "output"}.issubset(cols):
            prompt = self._alpaca_prompt()
            texts = []
            for i, o in zip(batch["input"], batch["output"]):
                texts.append(prompt.format(
                    instruction="Answer the following:",
                    input=i or "",
                    output=o or ""
                ) + EOS)
            return {"text": texts}

        raise ValueError(f"Pharma formatter can't infer columns: {sorted(list(cols))}")

    # ---------- Infer dataset format ----------
    def format_dataset(self, ds):
        print("✅ Inferring dataset format...")

        name = self.cfg.dataset_name
        cols = set(ds.column_names)

        # known dataset mappings
        if name == "tatsu-lab/alpaca":
            print("✅ Format: ALPACA")
            return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

        if name == "databricks/databricks-dolly-15k":
            print("✅ Format: DOLLY")
            return ds.map(self._format_dolly, batched=True, remove_columns=ds.column_names)

        if name == "anon8231489123/ShareGPT_Vicuna_unfiltered":
            print("✅ Format: SHAREGPT")
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if name == "OpenAssistant/oasst1":
            print("✅ Format: OASST (auto)")
            if "text" in cols:
                return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if name in ["sunny199/pharma-instruction-data", "pharma_instruction_data"]:
            print("✅ Format: PHARMA (custom)")
            return ds.map(self._format_pharma_custom, batched=True, remove_columns=ds.column_names)

        # fallback heuristics
        if "text" in cols:
            print("✅ Format: TEXT (fallback)")
            return ds.map(self._format_text, batched=True, remove_columns=ds.column_names)

        if "conversations" in cols or "messages" in cols:
            print("✅ Format: CHAT (fallback)")
            return ds.map(self._format_sharegpt, batched=True, remove_columns=ds.column_names)

        if {"instruction", "output"}.issubset(cols):
            print("✅ Format: ALPACA-LIKE (fallback)")
            return ds.map(self._format_alpaca, batched=True, remove_columns=ds.column_names)

        raise ValueError(f"❌ Could not infer dataset format. Columns: {sorted(list(cols))}")

    # ---------- Train ----------
    def train(self, ds):
        print("✅ Starting training...")
        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=ds,
            dataset_text_field="text",
            packing=self.cfg.packing,
            args=SFTConfig(
                per_device_train_batch_size=self.cfg.per_device_bs,
                gradient_accumulation_steps=self.cfg.grad_acc_steps,
                num_train_epochs=self.cfg.epochs,
                learning_rate=self.cfg.lr,
                warmup_ratio=self.cfg.warmup_ratio,
                optim="adamw_8bit",
                logging_steps=self.cfg.logging_steps,
                seed=self.cfg.seed,
                output_dir=self.cfg.output_dir,
                report_to="none",
            ),
        )
        trainer.train()

    # ---------- Save ----------
    def save(self):
        print("✅ Saving LoRA adapters to:", self.cfg.lora_save_path)
        self.model.save_pretrained(self.cfg.lora_save_path)
        self.tokenizer.save_pretrained(self.cfg.lora_save_path)

        if self.cfg.save_merged:
            print("✅ Merging LoRA + saving full model to:", self.cfg.merged_save_path)
            merged = self.model.merge_and_unload()
            merged.save_pretrained(self.cfg.merged_save_path, safe_serialization=True)
            self.tokenizer.save_pretrained(self.cfg.merged_save_path)

    # ---------- Full pipeline ----------
    def run(self):
        self.load_model()
        raw_ds = self.load_dataset()
        ds = self.format_dataset(raw_ds)
        print("✅ Sample formatted text:\n", ds["text"][0][:800])
        self.train(ds)
        self.save()
        print("✅ Done!")


# =========================
# 2) USE IT (Object)
# =========================
cfg = FineTuneConfig(
    model_name="unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    dataset_name="sunny199/pharma-instruction-data",
    split="train",
    lora_save_path="phi3_pharma_lora",
    output_dir="phi3_outputs",
    epochs=1,
    per_device_bs=2,
    grad_acc_steps=4,
    lr=2e-5,
    max_seq_length=4096,
    save_merged=False,  # True if you want merged fp16 model
)

trainer = UnslothFineTuner(cfg)
trainer.run()

✅ Loading model: unsloth/Phi-3-mini-4k-instruct-bnb-4bit
==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/5 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 1 | Total steps = 1
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,768,832 of 3,880,848,384 (1.54% trained)


Step,Training Loss


✅ Saving LoRA adapters to: phi3_pharma_lora
✅ Done!


## Running inference

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="phi3_pharma_lora",  # your lora_save_path
    max_seq_length=4096,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    "### Instruction:\nWhat is the dosage for amoxicillin?\n\n### Input:\n\n### Response:\n",
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
### Instruction:
What is the dosage for amoxicillin?

### Input:

### Response:
The dosage of amoxicillin varies depending on the condition being treated, the severity of the infection, and the patient's age and weight. Here are some general guidelines:

1. For adults with a mild to moderate infection:
   - The typical dosage is 500 mg to 875 mg every 12 hours or 2500 mg to 5250 mg per day in divided doses.

2. For children:
   - The dosage is usually calculated based on the child's weight. A common regimen is 20-50 mg/kg/day in divided dose

## Evaluation

In [6]:
!pip install -q rouge-score nltk bert-score evaluate sacrebleu

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.8 MB/s eta 0:00:00


In [7]:
import json
import time
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Optional

from unsloth import FastLanguageModel
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
import nltk
nltk.download("punkt", quiet=True)


@dataclass
class EvalConfig:
    # ── model to evaluate ──────────────────────────────────────
    lora_path: str = "phi3_pharma_lora"       # your lora_save_path
    base_model_name: Optional[str] = None      # if None, reads from adapter_config

    # ── generation ─────────────────────────────────────────────
    max_new_tokens: int = 256
    temperature: float = 0.1                   # low = deterministic = reproducible evals
    do_sample: bool = False                    # greedy by default for eval
    max_seq_length: int = 4096

    # ── metrics to run ─────────────────────────────────────────
    run_rouge: bool = True
    run_bertscore: bool = True                 # slow but gold-standard
    run_latency: bool = True
    run_qualitative: bool = True

    # ── test prompts ───────────────────────────────────────────
    # Replace these with your own domain prompts + expected answers
    test_cases: List[Dict] = field(default_factory=lambda: [
        {
            "instruction": "What is the recommended adult dosage for amoxicillin?",
            "input": "",
            "expected": "The typical adult dosage of amoxicillin is 250-500 mg every 8 hours or 500-875 mg every 12 hours, depending on the severity of infection.",
        },
        {
            "instruction": "List common side effects of ibuprofen.",
            "input": "",
            "expected": "Common side effects include stomach upset, nausea, heartburn, dizziness, and headache. Serious risks include GI bleeding and kidney damage with long-term use.",
        },
        {
            "instruction": "What drug class does metformin belong to?",
            "input": "",
            "expected": "Metformin belongs to the biguanide class of antidiabetic medications used as first-line treatment for type 2 diabetes.",
        },
        # Add more domain-specific test cases here …
    ])


class SLMEvaluator:
    """
    Industry-standard post-training evaluator for LoRA fine-tuned SLMs.
    Covers: ROUGE-L, BERTScore, latency benchmarking, and qualitative review.
    """

    def __init__(self, cfg: EvalConfig):
        self.cfg = cfg
        self.model = None
        self.tokenizer = None
        self.results: List[Dict] = []

    # ── Load ──────────────────────────────────────────────────────
    def load(self):
        print(f"✅ Loading fine-tuned model from: {self.cfg.lora_path}")
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.cfg.lora_path,
            max_seq_length=self.cfg.max_seq_length,
            load_in_4bit=True,
        )
        FastLanguageModel.for_inference(self.model)
        print("✅ Model ready for inference.\n")

    # ── Prompt formatting (mirrors your training format) ──────────
    def _build_prompt(self, instruction: str, input_text: str = "") -> str:
        return (
            "Below is an instruction that describes a task, paired with an input that provides further context.\n"
            "Write a response that appropriately completes the request.\n\n"
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            "### Response:\n"
        )

    # ── Single inference with timing ─────────────────────────────
    def _generate(self, prompt: str) -> tuple[str, float]:
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        t0 = time.perf_counter()
        with torch.no_grad():
            out_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.cfg.max_new_tokens,
                temperature=self.cfg.temperature,
                do_sample=self.cfg.do_sample,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        latency_ms = (time.perf_counter() - t0) * 1000

        # Decode only the newly generated tokens (strip the prompt)
        gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
        response = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        return response, latency_ms

    # ── ROUGE-L ──────────────────────────────────────────────────
    def _rouge_l(self, predictions: List[str], references: List[str]) -> Dict:
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
        scores = [scorer.score(ref, pred)["rougeL"].fmeasure
                  for pred, ref in zip(predictions, references)]
        return {
            "rougeL_mean": round(np.mean(scores), 4),
            "rougeL_min":  round(np.min(scores), 4),
            "rougeL_max":  round(np.max(scores), 4),
            "per_sample":  [round(s, 4) for s in scores],
        }

    # ── BERTScore ─────────────────────────────────────────────────
    def _bertscore(self, predictions: List[str], references: List[str]) -> Dict:
        print("   ⏳ Computing BERTScore (this takes ~30s on GPU)…")
        P, R, F1 = bert_score_fn(
            predictions, references,
            lang="en",
            device="cuda" if torch.cuda.is_available() else "cpu",
            verbose=False,
        )
        return {
            "bertscore_F1_mean": round(F1.mean().item(), 4),
            "bertscore_P_mean":  round(P.mean().item(), 4),
            "bertscore_R_mean":  round(R.mean().item(), 4),
            "per_sample_F1":     [round(v, 4) for v in F1.tolist()],
        }

    # ── Latency benchmark ─────────────────────────────────────────
    def _latency_benchmark(self, n_runs: int = 5) -> Dict:
        """
        Runs the first test prompt N times and reports p50/p90/p99 latency.
        Industry standard: use a fixed prompt, measure wall-clock time.
        """
        probe = self._build_prompt(
            self.cfg.test_cases[0]["instruction"],
            self.cfg.test_cases[0]["input"],
        )
        times = []
        for _ in range(n_runs):
            _, ms = self._generate(probe)
            times.append(ms)
        times_arr = np.array(times)
        return {
            "n_runs": n_runs,
            "p50_ms": round(float(np.percentile(times_arr, 50)), 1),
            "p90_ms": round(float(np.percentile(times_arr, 90)), 1),
            "p99_ms": round(float(np.percentile(times_arr, 99)), 1),
            "mean_ms": round(float(times_arr.mean()), 1),
        }

    # ── Main eval loop ────────────────────────────────────────────
    def evaluate(self) -> Dict:
        if self.model is None:
            self.load()

        predictions, references = [], []
        per_sample = []

        print(f"🔍 Running evaluation on {len(self.cfg.test_cases)} test cases…\n")

        for i, tc in enumerate(tqdm(self.cfg.test_cases, desc="Generating")):
            prompt = self._build_prompt(tc["instruction"], tc.get("input", ""))
            pred, latency_ms = self._generate(prompt)

            predictions.append(pred)
            references.append(tc["expected"])

            per_sample.append({
                "id": i,
                "instruction": tc["instruction"],
                "expected": tc["expected"],
                "predicted": pred,
                "latency_ms": round(latency_ms, 1),
            })

        self.results = per_sample

        # ── Aggregate metrics ──
        report = {"num_samples": len(self.cfg.test_cases)}

        if self.cfg.run_rouge:
            print("\n📊 Computing ROUGE-L…")
            report["rouge"] = self._rouge_l(predictions, references)

        if self.cfg.run_bertscore:
            print("📊 Computing BERTScore…")
            report["bertscore"] = self._bertscore(predictions, references)

        if self.cfg.run_latency:
            print("⏱️  Running latency benchmark…")
            report["latency"] = self._latency_benchmark()

        return report

    # ── Pretty print ──────────────────────────────────────────────
    def print_report(self, report: Dict):
        print("\n" + "="*60)
        print("          📋 EVALUATION REPORT")
        print("="*60)
        print(f"  Samples evaluated : {report['num_samples']}")

        if "rouge" in report:
            r = report["rouge"]
            print(f"\n  ROUGE-L (↑ better, 0–1)")
            print(f"    Mean : {r['rougeL_mean']}  |  Min: {r['rougeL_min']}  |  Max: {r['rougeL_max']}")

        if "bertscore" in report:
            b = report["bertscore"]
            print(f"\n  BERTScore F1 (↑ better, 0–1)")
            print(f"    Mean : {b['bertscore_F1_mean']}  |  P: {b['bertscore_P_mean']}  |  R: {b['bertscore_R_mean']}")

        if "latency" in report:
            l = report["latency"]
            print(f"\n  Latency (over {l['n_runs']} runs, same prompt)")
            print(f"    p50  : {l['p50_ms']} ms")
            print(f"    p90  : {l['p90_ms']} ms")
            print(f"    Mean : {l['mean_ms']} ms")

        print("\n" + "="*60)

    def print_qualitative(self):
        """Side-by-side predicted vs expected for every test case."""
        print("\n" + "="*60)
        print("          🔬 QUALITATIVE REVIEW")
        print("="*60)
        for r in self.results:
            print(f"\n[{r['id']+1}] {r['instruction']}")
            print(f"  EXPECTED  : {r['expected'][:300]}")
            print(f"  PREDICTED : {r['predicted'][:300]}")
            print(f"  Latency   : {r['latency_ms']} ms")
            print("-"*60)

    def save_results(self, path: str = "eval_results.json"):
        with open(path, "w") as f:
            json.dump(self.results, f, indent=2)
        print(f"\n💾 Per-sample results saved → {path}")

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.results)

In [9]:

PHARMA_ROLE = (
    "Your role as a doctor requires you to answer the medical questions "
    "taking into account the patient's description."
)

TEST_CASES = [
    # ── Dosage ──────────────────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "What is the standard adult dose for amoxicillin when treating a throat infection?",
        "expected": (
            "The standard adult dose of amoxicillin for a throat infection (pharyngitis/tonsillitis) "
            "is 500 mg three times daily for 10 days, or 875 mg twice daily for 10 days. "
            "Always complete the full course even if symptoms improve earlier."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "How fast does amlodipine start to work for high blood pressure?",
        "expected": (
            "Amlodipine begins lowering blood pressure within 6–12 hours of the first dose, "
            "with peak plasma concentrations reached in 6–12 hours after oral administration. "
            "However, the full antihypertensive effect may take 7–14 days of regular dosing."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "What is the correct metformin dose for a newly diagnosed type 2 diabetic patient?",
        "expected": (
            "Metformin is typically started at 500 mg once or twice daily with meals to minimize "
            "gastrointestinal side effects. The dose is gradually increased over 2–4 weeks to the "
            "usual maintenance dose of 1000 mg twice daily. Maximum dose is 2550 mg/day."
        ),
    },
    # ── Side effects ─────────────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "What are the most common side effects of taking ibuprofen regularly?",
        "expected": (
            "Common side effects of regular ibuprofen use include gastrointestinal upset (nausea, "
            "heartburn, stomach pain), headache, and dizziness. Long-term or high-dose use raises "
            "the risk of peptic ulcers, GI bleeding, kidney dysfunction, and increased cardiovascular "
            "events. It should always be taken with food and at the lowest effective dose."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "Can metformin cause vitamin B12 deficiency?",
        "expected": (
            "Yes, long-term metformin use is associated with reduced vitamin B12 absorption, "
            "occurring in approximately 10–30% of patients. This happens because metformin "
            "interferes with the calcium-dependent absorption of the B12-intrinsic factor complex "
            "in the ileum. Annual B12 monitoring is recommended for patients on long-term metformin."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "What side effects should I watch for when my patient starts atorvastatin?",
        "expected": (
            "The most important side effects to monitor with atorvastatin are myopathy and "
            "rhabdomyolysis (muscle pain, weakness, dark urine), elevated liver enzymes (check LFTs "
            "at baseline and if symptomatic), and new-onset diabetes. Common but less serious effects "
            "include headache, nausea, and joint pain. Advise patients to report unexplained muscle pain."
        ),
    },
    # ── Drug interactions ────────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "Is it safe to take ibuprofen and aspirin together?",
        "expected": (
            "Concurrent use of ibuprofen and aspirin is generally not recommended. Ibuprofen can "
            "competitively inhibit the irreversible binding of aspirin to COX-1, reducing aspirin's "
            "cardioprotective antiplatelet effect. If both are needed, aspirin should be taken at "
            "least 30 minutes before ibuprofen. Paracetamol is a safer alternative for pain relief "
            "in patients on low-dose aspirin."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "What happens if a patient takes warfarin and ciprofloxacin together?",
        "expected": (
            "Ciprofloxacin significantly potentiates the anticoagulant effect of warfarin by inhibiting "
            "CYP1A2 and CYP3A4 enzymes responsible for warfarin metabolism, and by reducing gut flora "
            "that produce vitamin K. This combination can lead to dangerously elevated INR and bleeding "
            "risk. INR should be monitored closely (within 3–5 days of starting ciprofloxacin) and the "
            "warfarin dose adjusted accordingly."
        ),
    },
    # ── Drug class / mechanism ───────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "What drug class does metformin belong to and how does it work?",
        "expected": (
            "Metformin belongs to the biguanide class of antidiabetic drugs. Its primary mechanism "
            "is the activation of AMP-activated protein kinase (AMPK), which decreases hepatic "
            "glucose production (gluconeogenesis), improves insulin sensitivity in peripheral tissues, "
            "and modestly reduces intestinal glucose absorption. It does not cause hypoglycaemia when "
            "used as monotherapy."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "How do ACE inhibitors lower blood pressure?",
        "expected": (
            "ACE inhibitors (e.g., lisinopril, ramipril) lower blood pressure by blocking "
            "angiotensin-converting enzyme, which prevents the conversion of angiotensin I to "
            "angiotensin II — a potent vasoconstrictor. The result is vasodilation, reduced aldosterone "
            "secretion (less sodium and water retention), and decreased cardiac afterload. "
            "They also increase levels of bradykinin, which contributes to their antihypertensive effect "
            "but also causes the characteristic dry cough."
        ),
    },
    # ── Contraindications ────────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "Can I prescribe metformin to a patient with stage 3b chronic kidney disease?",
        "expected": (
            "Metformin is contraindicated or requires dose reduction in CKD. Current guidelines "
            "allow metformin use with caution when eGFR is 30–44 mL/min/1.73m² (CKD 3b) with "
            "close monitoring, but it should be stopped if eGFR falls below 30. The risk is lactic "
            "acidosis due to metformin accumulation when renal clearance is impaired. "
            "Recheck eGFR every 3–6 months in these patients."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "Which patients should not take NSAIDs like diclofenac?",
        "expected": (
            "NSAIDs such as diclofenac are contraindicated in patients with: active peptic ulcer "
            "disease or GI bleeding history, severe renal or hepatic impairment, heart failure, "
            "established ischaemic heart disease or cerebrovascular disease (for high-dose long-term "
            "use), and the third trimester of pregnancy. Caution is also needed in elderly patients, "
            "those on anticoagulants, and patients with asthma (aspirin-exacerbated respiratory disease)."
        ),
    },
    # ── Pregnancy / special populations ─────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "Which antibiotics are safe to use during the first trimester of pregnancy?",
        "expected": (
            "Antibiotics generally considered safe in the first trimester include penicillins "
            "(amoxicillin, ampicillin), cephalosporins, and erythromycin. Avoid tetracyclines "
            "(tooth discoloration, bone development issues), fluoroquinolones (cartilage toxicity), "
            "metronidazole in the first trimester (teratogenicity concern, though evidence is limited), "
            "and trimethoprim (folate antagonist). Always use antibiotics only when clearly necessary "
            "and at the lowest effective dose."
        ),
    },
    {
        "instruction": PHARMA_ROLE,
        "input": "Is paracetamol safe during pregnancy and at what dose?",
        "expected": (
            "Paracetamol (acetaminophen) is considered the first-line analgesic and antipyretic "
            "during pregnancy and is generally regarded as safe at all stages when used at the "
            "recommended dose of 500–1000 mg every 4–6 hours, not exceeding 4 g/day. It should "
            "be used at the lowest effective dose for the shortest duration. Recent observational "
            "studies suggest possible associations with neurodevelopmental outcomes with prolonged "
            "use, so unnecessary long-term use should be avoided."
        ),
    },
    # ── Storage / formulation ────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "How should insulin be stored at home?",
        "expected": (
            "Unopened insulin vials or pens should be refrigerated at 2–8°C and never frozen. "
            "Once opened, most insulin formulations can be kept at room temperature (below 25–30°C) "
            "for 28 days (check the specific product's label, as it varies). Opened vials should "
            "not be refrigerated after the first use according to some guidelines. "
            "Insulin should be protected from direct sunlight and heat, and never used if it "
            "appears cloudy, discolored, or contains particles (for clear insulins)."
        ),
    },
    # ── OTC / self-medication ────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "A patient asks if they can take cetirizine and loratadine together for allergies.",
        "expected": (
            "Taking cetirizine and loratadine together is not recommended. Both are second-generation "
            "H1 antihistamines and combining two antihistamines of the same class does not provide "
            "additional benefit but increases the risk of side effects such as excessive sedation, "
            "dry mouth, urinary retention, and blurred vision. If one antihistamine does not provide "
            "adequate relief, switching to a different class or adding a nasal corticosteroid is a "
            "more appropriate approach."
        ),
    },
    # ── Overdose / toxicity ──────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "What are the signs of paracetamol overdose and how is it treated?",
        "expected": (
            "Paracetamol overdose presents in four stages: (1) nausea, vomiting, malaise within 24h; "
            "(2) apparent clinical improvement but rising LFTs (24–72h); (3) peak hepatotoxicity — "
            "jaundice, coagulopathy, encephalopathy (72–96h); (4) recovery or fulminant liver failure. "
            "Treatment: activated charcoal if within 1–2 hours of ingestion, then IV N-acetylcysteine "
            "(NAC) based on the Rumack-Matthew nomogram. Early NAC is highly effective; delayed "
            "treatment significantly worsens prognosis."
        ),
    },
    # ── Pharmacokinetics ────────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "What is the half-life of diazepam and why does it matter clinically?",
        "expected": (
            "Diazepam has a very long half-life of 20–100 hours, with its active metabolite "
            "desmethyldiazepam adding further duration (36–200 hours). This matters clinically "
            "because: (1) drug accumulates with repeated dosing, especially in the elderly and those "
            "with hepatic impairment; (2) once-daily dosing is possible; (3) withdrawal after chronic "
            "use must be tapered very slowly to avoid seizures; (4) effects can persist well after "
            "the last dose, affecting driving and cognitive function."
        ),
    },
    # ── Patient counselling ──────────────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "What should I tell a patient who is just starting on lisinopril?",
        "expected": (
            "Key counselling points for lisinopril: (1) Take at the same time each day, with or "
            "without food. (2) A dry cough is common (10–15% of patients) — if intolerable, an ARB "
            "like losartan can be substituted. (3) Dizziness when standing (orthostatic hypotension) "
            "may occur especially in the first days — rise slowly. (4) Avoid NSAIDs, potassium "
            "supplements, and potassium-sparing diuretics without medical advice. (5) Seek immediate "
            "help for swelling of face, lips, or throat (angioedema — rare but serious). "
            "(6) Do not stop abruptly. Blood pressure will be monitored at follow-up."
        ),
    },
    # ── Pharmacogenomics / resistance ────────────────────────────────────────
    {
        "instruction": PHARMA_ROLE,
        "input": "Why do some patients not respond to clopidogrel as an antiplatelet agent?",
        "expected": (
            "Clopidogrel is a prodrug that requires activation by the liver enzyme CYP2C19. "
            "Approximately 25–30% of patients carry loss-of-function CYP2C19 alleles (*2, *3), "
            "making them 'poor metabolizers' who cannot adequately convert clopidogrel to its "
            "active thiol metabolite. This results in reduced platelet inhibition and higher risk "
            "of cardiovascular events (stent thrombosis, MI). Alternatives include prasugrel or "
            "ticagrelor, which do not require CYP2C19 activation. Pharmacogenomic testing can "
            "identify these patients before initiating therapy."
        ),
    },
]
eval_cfg = EvalConfig(
    lora_path="phi3_pharma_lora",      # ← same as your cfg.lora_save_path
    max_new_tokens=256,
    run_rouge=True,
    run_bertscore=True,                # set False to skip (saves ~30s)
    run_latency=True,
    test_cases=TEST_CASES
)

evaluator = SLMEvaluator(eval_cfg)
report = evaluator.evaluate()

evaluator.print_report(report)
evaluator.print_qualitative()
evaluator.save_results("eval_results.json")

# View as a DataFrame in Colab
df = evaluator.to_dataframe()
display(df[["id", "instruction", "expected", "predicted", "latency_ms"]])


✅ Loading fine-tuned model from: phi3_pharma_lora
==((====))==  Unsloth 2026.6.9: Fast Mistral patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Model ready for inference.

🔍 Running evaluation on 20 test cases…



Generating: 100%|██████████| 20/20 [05:40<00:00, 17.05s/it]



📊 Computing ROUGE-L…
📊 Computing BERTScore…
   ⏳ Computing BERTScore (this takes ~30s on GPU)…


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


⏱️  Running latency benchmark…

          📋 EVALUATION REPORT
  Samples evaluated : 20

  ROUGE-L (↑ better, 0–1)
    Mean : 0.1768  |  Min: 0.1294  |  Max: 0.2857

  BERTScore F1 (↑ better, 0–1)
    Mean : 0.8527  |  P: 0.8378  |  R: 0.8685

  Latency (over 5 runs, same prompt)
    p50  : 16745.7 ms
    p90  : 16778.6 ms
    Mean : 16535.6 ms


          🔬 QUALITATIVE REVIEW

[1] Your role as a doctor requires you to answer the medical questions taking into account the patient's description.
  EXPECTED  : The standard adult dose of amoxicillin for a throat infection (pharyngitis/tonsillitis) is 500 mg three times daily for 10 days, or 875 mg twice daily for 10 days. Always complete the full course even if symptoms improve earlier.
  PREDICTED : The standard adult dose for amoxicillin when treating a throat infection is typically 500 mg taken three times a day for 7 to 14 days. However, the exact dosage may vary based on the severity of the infection, the patient's medical history, and

,id,instruction,expected,predicted,latency_ms
0,0,Your role as a doctor requires you to answer t...,The standard adult dose of amoxicillin for a t...,The standard adult dose for amoxicillin when t...,22362.7
1,1,Your role as a doctor requires you to answer t...,Amlodipine begins lowering blood pressure with...,"Amlodipine, a calcium channel blocker, typical...",16330.6
2,2,Your role as a doctor requires you to answer t...,Metformin is typically started at 500 mg once ...,The initial dose of metformin for a newly diag...,16234.9
3,3,Your role as a doctor requires you to answer t...,Common side effects of regular ibuprofen use i...,The most common side effects of taking ibuprof...,16680.4
4,4,Your role as a doctor requires you to answer t...,"Yes, long-term metformin use is associated wit...","Yes, long-term use of metformin has been assoc...",16002.3
5,5,Your role as a doctor requires you to answer t...,The most important side effects to monitor wit...,When your patient begins treatment with atorva...,19813.6
6,6,Your role as a doctor requires you to answer t...,Concurrent use of ibuprofen and aspirin is gen...,"As a doctor, I would advise against taking ibu...",16309.2
7,7,Your role as a doctor requires you to answer t...,Ciprofloxacin significantly potentiates the an...,"When a patient takes warfarin, an anticoagulan...",16603.9
8,8,Your role as a doctor requires you to answer t...,Metformin belongs to the biguanide class of an...,Metformin belongs to the drug class known as b...,17099.6
9,9,Your role as a doctor requires you to answer t...,"ACE inhibitors (e.g., lisinopril, ramipril) lo...","ACE inhibitors, or Angiotensin-Converting Enzy...",16763.7
